# 领域权重重新评估

## 结论

0.65 不是已证明合理的默认值。当前标签混合内容领域与数据来源；本次只检查定义和量级，不启动训练。

## 数据与口径

396,266,210 条历史 v7 样本，尚未应用单字过滤。下文计算仅涉及领域权重，排除位置权重；名义加权份额不等于经过长度分桶、裁剪和 AdamW 后的实际参数影响。

In [1]:
from pathlib import Path
import json, math, sys
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'training/train_sharded.py').exists())
path=root/'training/data/processed/padding-fixed-web-350m-v7-20260925/manifest.json'
m=json.loads(path.read_text()); c=m['statistics']['domain_samples']; n=sum(c.values())
assert n==m['statistics']['samples']==396266210
print(c)

{'news': 7787, 'academic': 9985, 'encyclopedia': 91670, 'dialogue': 79802, 'wikipedia': 21365413, 'synthetic_multistyle': 24711553, 'web': 350000000}


## 方法

每条样本的原始权重为 N_d 的负 α 次方，再归一化为全体样本平均权重 1。领域名义份额 q_d=N_d^(1−α)/ΣN_j^(1−α)。α=0 表示显式领域权重相同，α=1 表示各标签名义总份额相同。

In [2]:
a=.65; z=sum(v**(1-a) for v in c.values()); expected=[v**(-a)/(z/n) for v in c.values()]
sys.path[:0]=[str(root/'training/.deps'),str(root/'training')]
from train_sharded import domain_weight_table
actual=domain_weight_table(m,list(c),a)
assert all(math.isclose(x,y,rel_tol=1e-12) for x,y in zip(expected,actual))
assert math.isclose(sum(v*w for v,w in zip(c.values(),actual))/n,1)
for (d,v),w in zip(c.items(),actual): print(d,v,100*v/n,w,100*v*w/n)

news 7787 0.0019650931125315984 620.5301040349854 1.219399433557666
academic 9985 0.0025197707369497896 527.9310724381861 1.3302652674562607
encyclopedia 91670 0.02313343850337378 124.9412275634775 2.890320204375736
dialogue 79802 0.020138482158244077 136.72392416609952 2.7534123074241115
wikipedia 21365413 5.391681768677677 3.6127767805323203 19.47894270189815
synthetic_multistyle 24711553 6.236098959838135 3.286763408299468 20.496581871730356
web 350000000 88.32446248697309 0.586825854968574 51.831078213557724


## 敏感性与标签划分

以下为确定性公式推导，不是模型实验。分组变动示例只重命名网页来源，不增加或减少文本。

In [3]:
for a in (0,.25,.3,.5,.65,1):
 z=sum(v**(1-a) for v in c.values())
 print(a,(c['web']/c['news'])**a,c['web']**(1-a)/z)
a=.65; web=c['web']**(1-a); other=sum(v**(1-a) for d,v in c.items() if d!='web')
print('web before/after splitting into 10 equally sized labels:',web/(web+other),web*10**a/(web*10**a+other))

0 1.0 0.8832446248697309
0.25 14.560438938915771 0.7908840831975383
0.3 24.877650423603 0.7664233625616583
0.5 212.00638209389462 0.6434478312363846
0.65 1057.4348399632397 0.5183107821355772
1 44946.706048542444 0.14285714285714285
web before/after splitting into 10 equally sized labels: 0.5183107821355772 0.8277773653423879


## 解释、限制与下一步

少数来源不是高质量的同义词。关闭领域权重也不保证更好，因为网页原始样本份额为 88.3%，而现有语料配比本身经过人为采集和截取。代码与来源映射显示 news/academic 与 web/wikipedia 不属于互斥语义分类；现有位置消融固定领域权重，来源对比同时改变语料和权重，不能识别领域权重的独立效果。

建议后续先比较 α=0 与 α=0.65，关闭位置权重、过滤单字，保持初始化、样本、batch、学习率和预算一致；用验证集选择，test 只在最终方案确定后评估。本次不执行训练。

In [4]:
f=json.loads((root/'training/reviews/domain-weight-review-20261004/findings.json').read_text())
assert f['population']==n
assert math.isclose(f['news_to_web_ratio'],actual[0]/actual[-1],rel_tol=1e-12)
assert math.isclose(sum(r['weighted_share'] for r in f['domains']),1)
print('All formula and trainer checks passed.')

All formula and trainer checks passed.
